# Grokking Reproduction — Nanda et al. (2023)
**Paper:** Progress Measures for Grokking via Mechanistic Interpretability  
**DOI:** 10.48550/arXiv.2301.05217  

Run cells in order. Each cell is independent and re-runnable after a `git pull`.

In [ ]:
# Cell 0 — Colab setup (run once per session)
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/MrFaruk0/grokking-reproduction.git"
REPO_DIR = Path("/content/grokking-reproduction")

if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.chdir(REPO_DIR)
    os.system("git pull")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
os.system("pip install -r requirements.txt -q")
Path("outputs").mkdir(exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)
print("Setup complete.")

In [ ]:
# Cell 1 — Read PROJECT_MEMORY.md
print(Path("PROJECT_MEMORY.md").read_text())

In [ ]:
# Cell 2 — Environment check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3 — Generate data
from src.data import generate_dataset, describe_dataset
train_data, test_data = generate_dataset(p=113, train_fraction=0.3, seed=42)
describe_dataset(train_data, test_data, p=113)

In [ ]:
# Cell 4 — Initialize model
from src.model import GrokkingTransformer
model = GrokkingTransformer(p=113, d_model=128, num_heads=4, d_mlp=512, num_layers=1)
print(f"Parameters: {model.count_parameters():,}")

In [ ]:
# Cell 5 — Train (≈ 15–30 min on T4 GPU)
from src.train import train
history = train(
    model=model, train_data=train_data, test_data=test_data,
    epochs=20000, lr=1e-3, weight_decay=1.0,
    beta1=0.9, beta2=0.98,
    log_every=100,
    checkpoint_dir=Path("checkpoints"),
    checkpoint_every=1000,
    log_path=Path("outputs/training_log.csv"),
)

In [ ]:
# Cell 6 — Figure 1: Grokking curve
from src.figures import plot_grokking_curve
plot_grokking_curve(history, save_dir=Path("outputs"))
from IPython.display import Image
Image("outputs/figure1_grokking_curve.png")

In [ ]:
# Cell 7 — Figure 2: Progress measures
from src.figures import plot_progress_measures
plot_progress_measures(history, save_dir=Path("outputs"))
from IPython.display import Image
Image("outputs/figure2_progress_measures.png")

In [ ]:
# Cell 8 — Figure 3: Fourier analysis of W_E
from src.analysis import compute_fourier_spectrum
from src.figures import plot_fourier_spectrum
spectrum = compute_fourier_spectrum(model, p=113)
plot_fourier_spectrum(spectrum, save_dir=Path("outputs"))
from IPython.display import Image
Image("outputs/figure3_fourier_embeddings.png")

In [ ]:
# Cell 9 — Figure 4: Weight norm over training
from src.figures import plot_weight_norm
plot_weight_norm(history, save_dir=Path("outputs"))
from IPython.display import Image
Image("outputs/figure4_weight_norm.png")

In [ ]:
# Cell 10 — Further Exploration 1: weight decay sweep (≈ 1–2 h on T4)
from src.train import sweep_weight_decay
from src.figures import plot_wd_sweep
wd_results = sweep_weight_decay(
    weight_decays=[0.0, 0.1, 0.5, 1.0, 2.0],
    p=113, epochs=20000, seed=42,
    log_dir=Path("outputs/wd_sweep"),
)
plot_wd_sweep(wd_results, save_dir=Path("outputs"))
from IPython.display import Image
Image("outputs/figure5_wd_sweep.png")

In [ ]:
# Cell 11 — Further Exploration 2: varying p (≈ 1–2 h on T4)
from src.train import sweep_prime_p
from src.figures import plot_p_sweep
p_results = sweep_prime_p(
    primes=[53, 97, 113, 127],
    epochs=20000, seed=42,
    log_dir=Path("outputs/p_sweep"),
)
plot_p_sweep(p_results, save_dir=Path("outputs"))
from IPython.display import Image
Image("outputs/figure6_p_sweep.png")

In [ ]:
# Cell 12 — Further Exploration 3: operations (≈ 1 h on T4)
from src.train import sweep_operations
from src.figures import plot_operations_comparison
op_results = sweep_operations(
    operations=["add", "subtract", "multiply"],
    p=113, epochs=20000, seed=42,
    log_dir=Path("outputs/op_sweep"),
)
plot_operations_comparison(op_results, save_dir=Path("outputs"))
from IPython.display import Image
Image("outputs/figure7_operations.png")

In [ ]:
# Cell 13 — Further Exploration 4: model depth (≈ 1 h on T4)
from src.train import sweep_depth
from src.figures import plot_depth_comparison
depth_results = sweep_depth(
    depths=[1, 2, 3],
    p=113, epochs=20000, seed=42,
    log_dir=Path("outputs/depth_sweep"),
)
plot_depth_comparison(depth_results, save_dir=Path("outputs"))
from IPython.display import Image
Image("outputs/figure8_depth.png")

In [ ]:
# Cell 14 — Update PROJECT_MEMORY.md with all results
from src.utils import update_memory
update_memory(
    history,
    wd_results=wd_results,
    p_results=p_results,
    op_results=op_results,
    depth_results=depth_results,
)
print(Path("PROJECT_MEMORY.md").read_text())